In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/Volumes/Projets_Dev/Projets_Dev/Data_Analyst_Portfolio/Projet_1_Analyse_Ventes")
sql_path = PROJECT_ROOT / "sql" / "init_duckdb.sql"

con = duckdb.connect()
con.execute(f"SET file_search_path='{PROJECT_ROOT.as_posix()}';")
con.execute(sql_path.read_text())

 TABLE 1 — KPI mensuel

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE kpi_monthly AS
SELECT
    date_trunc('month', o.order_purchase_timestamp) AS month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS revenue,
    ROUND(
        SUM(oi.price + oi.freight_value) / COUNT(DISTINCT o.order_id),
        2
    ) AS avg_order_value
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY 1
ORDER BY 1;
""")

In [ ]:
con.execute("""
SELECT *
FROM kpi_monthly
LIMIT 10;
""").fetchdf()

In [ ]:
con.execute("""
COPY kpi_monthly
TO 'data_clean/kpi_monthly.csv'
WITH (HEADER, SEP ',');
""")

TABLE 2 — Catégories

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE CATEGORIES AS
SELECT
      COALESCE(P.product_category_name, 'unknown') AS product_category_name,
      SUM(O.price + O.freight_value) AS revenue,
      COUNT(*) AS items_sold
FROM products AS P
JOIN order_items AS O
   ON O.product_id = P.product_id
JOIN orders AS OD
   ON OD.order_id = O.order_id
   WHERE OD.order_status = 'delivered'
GROUP BY
            1
ORDER BY 2 DESC
LIMIT 10;

""")

In [ ]:
con.execute("""
SELECT *
FROM categories""").fetchdf()

In [ ]:
con.execute("""
COPY CATEGORIES
TO 'data_clean/categories.CSV'
WITH (HEADER, SEP ',');
""")

TABLE 3 — Satisfaction

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE satisfaction AS
SELECT
      O.product_id AS product_id,
      ROUND(AVG(R.review_score), 2) AS avg_score,
      COUNT(*) AS reviews_count,
      ROUND((SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)), 2) AS bad_review_rate
FROM reviews AS R
JOIN order_items AS O
    ON R.order_id = O.order_id
JOIN orders AS ORD
   ON ORD.order_id = O.order_id
WHERE order_status = 'delivered'
GROUP BY O.product_id
HAVING COUNT(*)  >= 50
ORDER BY AVG(R.review_score) desc
; """)

In [ ]:
con.execute("""
SELECT *
FROM satisfaction
            """).fetchdf()

In [ ]:
con.execute("""
COPY satisfaction TO 'data_clean/satisfaction.csv' WITH(HEADER, SEP ',');
""")

TABLE 4 - Geography 

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE kpi_geography AS
SELECT
    COALESCE(c.customer_state, 'unknown') AS customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS revenue
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY 1
ORDER BY revenue DESC;

            """).fetchdf()

In [ ]:
con.execute("""
COPY kpi_geography
TO 'data_clean/kpi_geography.csv'
WITH (HEADER, SEP ',');
            """)